# CNN-EXP-029 — Signed Relative-Position Cross-Attention, 32-bp head only — 10k screen

This fixes the central weakness revealed by EXP028.

EXP028 used content-only cross-attention:

```text
score(i,j) = Q_i · K_j / sqrt(d)
```

It had no explicit knowledge that a context anchor was, for example:

```text
-256 bp
+1 kb
-4 kb
```

from the current 32-bp query region. Its attention therefore stayed close to
uniform over the full global window.

EXP029 uses:

```text
score_h(i,j) =
    Q_h(i) · K_h(j) / sqrt(d)
    + B_h(Δbp(i,j))
```

where `B_h` is a **learned signed relative-position bias for each attention head**.

## Relative-position parameterization

- signed distance: `context_position - query_center`;
- 128-bp distance bins;
- separate left/upstream and right/downstream bins;
- maximum represented magnitude = 6400 bp;
- 101 total relative-position buckets;
- each of 4 attention heads learns its own bias curve;
- bias starts at zero, so the model starts equivalent to content-only attention.

This allows different heads to specialize, for example:

```text
head 1 → local ±256 bp
head 2 → 0.5–2 kb left
head 3 → long-range right context
head 4 → broad background context
```

## Core architecture

```text
BPE200 global window = 14,336 bp
        │
        ├── shallow stem on regular 16-bp anchors → K / V
        │
        └── strong EXP020 dilated CNN → 64 region queries
                                      ↓
                          positional cross-attention
                                      ↓
final logit = baseline CNN logit
            + learnable_gate × attention_delta
```

Only the **32-bp** regional objective is optimized.

## Training

```text
10,000 steps
checkpoint every 2,500
full holdout @10k
```

This is an architecture screen. If it is promising, the same architecture can be
run to 50–75k later.

## Context diagnostics

For positive 32-bp regions the notebook reports:

- attention mass within ±128/256/512/1k/2k/4k/6k bp;
- the same mass expected under uniform attention;
- enrichment/depletion versus uniform;
- effective radius containing 50/80/90/95% of attention;
- normalized attention entropy;
- left/right mass;
- edge-of-window mass;
- the learned signed relative-position bias curve for every head.

Now the effective-radius diagnostic can meaningfully answer whether useful context
should probably be **smaller or larger** than the current ±6144 bp.


In [1]:
from pathlib import Path
from collections import OrderedDict
import gc
import hashlib
import math
import re
import sys
import time
import warnings

from IPython.display import display
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    dml_weighted_profile_bce_with_logits,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()
print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = tuple(sorted(set(split.intervals("train")["contig"].astype(str))))
validation_contigs = tuple(sorted(set(split.intervals("validation")["contig"].astype(str))))
assert set(train_contigs).isdisjoint(validation_contigs)

profile_config = ProfileWindowConfig(
    target_length=2048,
    context_flank=130,
    positive_branch_fraction=0.5,
)

GLOBAL_FLANK = 6144
TARGET_LENGTH = profile_config.target_length
LOCAL_FLANK = profile_config.context_flank
GLOBAL_LENGTH = GLOBAL_FLANK + TARGET_LENGTH + GLOBAL_FLANK
LOCAL_LENGTH = LOCAL_FLANK + TARGET_LENGTH + LOCAL_FLANK
LOCAL_START_IN_GLOBAL = GLOBAL_FLANK - LOCAL_FLANK
LOCAL_END_IN_GLOBAL = LOCAL_START_IN_GLOBAL + LOCAL_LENGTH

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

cache_contigs = tuple(dict.fromkeys((*train_contigs, *validation_contigs)))

assert GLOBAL_LENGTH == 14336
# assert LOCAL_LENGTH == 1284
assert LOCAL_END_IN_GLOBAL - LOCAL_START_IN_GLOBAL == LOCAL_LENGTH

display(split.summary)
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Cache contigs:", len(cache_contigs))
print("Local / Global:", LOCAL_LENGTH, "/", GLOBAL_LENGTH)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Cache contigs: 12
Local / Global: 2308 / 14336


In [4]:
# ============================================================
# BPE-200 CONFIGURATION
# ============================================================

BPE_BASE_MERGES = 100
BPE_MERGES = 200

BPE_BASE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_BASE_MERGES}.pt"
)

BPE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_MERGES}.pt"
)

BUILD_BPE_IF_MISSING = True

print("Base artifact:", BPE_BASE_PATH)
print("Target artifact:", BPE_PATH)
print(f"BPE-{BPE_MERGES} exists:", BPE_PATH.is_file())

if BPE_MERGES >= 251:
    raise ValueError(
        "Этот notebook использует uint8 token ids; "
        "для >=251 merges нужен более широкий dtype."
    )


Base artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_100.pt
Target artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_200.pt
BPE-200 exists: True


In [5]:
# ============================================================
# ONE-TIME STEP: EXTEND BPE-100 -> BPE-200
# ============================================================
# Merge rules are learned ONLY from train contigs.
# Validation sequence is never used to select BPE rules.


def _norm_rules(history):
    return [tuple(int(v) for v in rule) for rule in history]


def _merge_tokens_cpu(
    tokens: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Apply one BPE rule exactly left-to-right on a 1D CPU tensor."""

    if tokens.device.type != "cpu":
        raise ValueError("_merge_tokens_cpu expects a CPU tensor")

    if tokens.numel() < 2:
        return tokens

    matches = (
        (tokens[:-1] == left)
        & (tokens[1:] == right)
    )

    starts = torch.nonzero(
        matches,
        as_tuple=False,
    ).flatten()

    if starts.numel() == 0:
        return tokens

    # Overlapping candidates are possible only for left == right.
    # Keep every second candidate inside each consecutive run.
    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor(
                [True],
                dtype=torch.bool,
            ),
            starts[1:] != starts[:-1] + 1,
        ))

        run_ids = (
            torch.cumsum(
                new_run.to(torch.int64),
                dim=0,
            )
            - 1
        )

        run_starts = torch.nonzero(
            new_run,
            as_tuple=False,
        ).flatten()

        offsets = (
            torch.arange(
                starts.numel(),
                dtype=torch.int64,
            )
            - run_starts[run_ids]
        )

        starts = starts[
            offsets.remainder(2) == 0
        ]

    out = tokens.clone()
    out[starts] = new_token

    remove = torch.zeros(
        tokens.numel(),
        dtype=torch.bool,
    )
    remove[starts + 1] = True

    return out[~remove].contiguous()


def _apply_rules_cpu(
    tokens: torch.Tensor,
    rules,
):
    for left, right, new_token in rules:
        tokens = _merge_tokens_cpu(
            tokens,
            left,
            right,
            new_token,
        )
    return tokens


def build_bpe_target_from_base():

    if not BPE_BASE_PATH.is_file():
        raise FileNotFoundError(
            f"Нужен готовый BPE-{BPE_BASE_MERGES}: "
            f"{BPE_BASE_PATH}"
        )

    base = torch.load(
        BPE_BASE_PATH,
        map_location="cpu",
        weights_only=False,
    )

    rules = _norm_rules(
        base["merge_history"]
    )

    if len(rules) != BPE_BASE_MERGES:
        raise ValueError(
            f"Expected BPE-{BPE_BASE_MERGES}, "
            f"got {len(rules)} merges"
        )

    token_text = {
        int(k): str(v)
        for k, v in base["token_text"].items()
    }

    token_length = {
        int(k): int(v)
        for k, v in base["token_length"].items()
    }

    # Replay the existing BPE-100 once on TRAIN contigs.
    # These compact uint8 streams are then used to learn merges 101..200.
    streams = {}
    total_bp = 0
    started = time.perf_counter()

    for i, contig in enumerate(
        train_contigs,
        start=1,
    ):
        raw = torch.as_tensor(
            sequences[contig],
            dtype=torch.uint8,
        )

        tokens = torch.where(
            raw < 8,
            torch.remainder(raw, 4),
            torch.full_like(raw, 4),
        ).to(torch.uint8)

        tokens = _apply_rules_cpu(
            tokens,
            rules,
        ).to(torch.uint8).contiguous()

        streams[contig] = tokens
        total_bp += raw.numel()

        print(
            f"BPE-{BPE_BASE_MERGES} replay "
            f"{i:>3}/{len(train_contigs)} "
            f"{contig}: {tokens.numel():,} tokens"
        )

    PAIR_BASE = 256
    PAIR_SPACE = PAIR_BASE * PAIR_BASE

    while len(rules) < BPE_MERGES:

        counts = torch.zeros(
            PAIR_SPACE,
            dtype=torch.int64,
        )

        for tokens in streams.values():

            if tokens.numel() < 2:
                continue

            pair_code = (
                tokens[:-1].to(torch.int64)
                * PAIR_BASE
                + tokens[1:].to(torch.int64)
            )

            counts += torch.bincount(
                pair_code,
                minlength=PAIR_SPACE,
            )

        best = int(
            torch.argmax(counts)
        )

        best_count = int(
            counts[best]
        )

        left, right = divmod(
            best,
            PAIR_BASE,
        )

        new_token = 5 + len(rules)

        if best_count <= 0:
            raise RuntimeError(
                "No pair left to merge"
            )

        if new_token >= 256:
            raise RuntimeError(
                "uint8 token storage is no longer sufficient"
            )

        for contig in streams:
            streams[contig] = _merge_tokens_cpu(
                streams[contig],
                left,
                right,
                new_token,
            ).to(torch.uint8).contiguous()

        rules.append(
            (left, right, new_token)
        )

        token_text[new_token] = (
            token_text[left]
            + token_text[right]
        )

        token_length[new_token] = (
            token_length[left]
            + token_length[right]
        )

        total_tokens = sum(
            t.numel()
            for t in streams.values()
        )

        print(
            f"merge={len(rules):3d} "
            f"pair=({left},{right}) -> {new_token} "
            f"count={best_count:,} "
            f"compression={total_bp / total_tokens:.4f}x"
        )

    artifact = {
        "merge_history": rules,
        "token_text": token_text,
        "token_length": token_length,
        "vocab_size": 5 + len(rules),
        "num_merges": len(rules),
        "train_bp": int(total_bp),
        "source_artifact": str(BPE_BASE_PATH),
        "case_independent": True,
        "softmask_mode": "per-occurrence lowercase_count",
    }

    torch.save(
        artifact,
        BPE_PATH,
    )

    print()
    print("Saved:", BPE_PATH)
    print(
        "Minutes:",
        round(
            (time.perf_counter() - started) / 60,
            2,
        ),
    )


if not BPE_PATH.is_file():

    if BUILD_BPE_IF_MISSING:
        build_bpe_target_from_base()
    else:
        raise FileNotFoundError(
            BPE_PATH
        )

else:
    print(
        f"Using existing BPE-{BPE_MERGES} artifact"
    )


Using existing BPE-200 artifact


In [6]:
# ============================================================
# LOAD BPE-200 + ARTIFACT INTEGRITY
# ============================================================

bpe_artifact = torch.load(
    BPE_PATH,
    map_location="cpu",
    weights_only=False,
)

merge_history = [
    tuple(int(v) for v in rule)
    for rule in bpe_artifact["merge_history"]
]

token_text = {
    int(k): str(v)
    for k, v in bpe_artifact["token_text"].items()
}

token_length = {
    int(k): int(v)
    for k, v in bpe_artifact["token_length"].items()
}

vocab_size = int(
    bpe_artifact["vocab_size"]
)

assert len(merge_history) == BPE_MERGES
assert vocab_size == 5 + BPE_MERGES
assert set(range(vocab_size)).issubset(token_length)

# BPE-200 must be a strict continuation of the exact BPE-100 artifact.
base_artifact = torch.load(
    BPE_BASE_PATH,
    map_location="cpu",
    weights_only=False,
)

base_rules = [
    tuple(int(v) for v in rule)
    for rule in base_artifact["merge_history"]
]

assert len(base_rules) == BPE_BASE_MERGES
assert merge_history[:BPE_BASE_MERGES] == base_rules

for merge_index, (
    left,
    right,
    new_token,
) in enumerate(merge_history):

    assert new_token == 5 + merge_index

    assert (
        token_length[new_token]
        == token_length[left]
        + token_length[right]
    )

    assert (
        token_text[new_token]
        == token_text[left]
        + token_text[right]
    )

print(
    f"BPE-{BPE_MERGES} artifact integrity: OK"
)

PAD_TOKEN_ID = vocab_size
N_TOKEN_ID = 4

MAX_TOKEN_LENGTH = max(
    token_length.values()
)

if MAX_TOKEN_LENGTH >= 256:
    raise ValueError(
        "uint8 lowercase_count requires "
        "max token length < 256 bp"
    )

TOKEN_LENGTH_LOOKUP = torch.zeros(
    vocab_size,
    dtype=torch.int16,
)

for token_id, length in token_length.items():
    TOKEN_LENGTH_LOOKUP[token_id] = length

MERGE_HASH = hashlib.sha1(
    repr(merge_history).encode()
).hexdigest()[:12]

CACHE_FORMAT_VERSION = 2
CACHE_ENGINE = "torch_cpu_exact_left_to_right"

# Separate cache: BPE-100 cache can never be reused accidentally.
CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / (
        f"bpe{BPE_MERGES}_pretokenized_"
        f"v{CACHE_FORMAT_VERSION}_{MERGE_HASH}"
    )
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("BPE merges:", len(merge_history))
print("Vocab:", vocab_size)
print("Max token length:", MAX_TOKEN_LENGTH)
print("Merge hash:", MERGE_HASH)
print("Cache engine:", CACHE_ENGINE)
print("Cache:", CACHE_DIR)


BPE-200 artifact integrity: OK
BPE merges: 200
Vocab: 205
Max token length: 8
Merge hash: 03d3d5e77ea4
Cache engine: torch_cpu_exact_left_to_right
Cache: D:\Projects\EPIC\EPIC\artifacts\bpe200_pretokenized_v2_03d3d5e77ea4


In [7]:
# ============================================================
# EXACT ONE-TIME CPU PRETOKENIZATION — BPE-200
# ============================================================
# CPU is intentional here. The old full-contig DirectML path produced
# invalid no-merge caches on long contigs. Training itself remains GPU-based.

INDEX_CHUNK_TOKENS = 1_000_000


def reverse_complement_storage_codes_cpu(raw: torch.Tensor) -> torch.Tensor:
    if raw.device.type != "cpu":
        raise ValueError("reverse_complement_storage_codes_cpu expects CPU")
    x = raw.to(torch.int16)
    is_base = x < 8
    case = ((x >= 4) & (x < 8)).to(torch.int16) * 4
    rc = 3 - torch.remainder(x, 4) + case
    rc = torch.where(is_base, rc, torch.full_like(rc, 8))
    return rc.flip(0).to(torch.uint8).contiguous()


def _merge_tokens_and_lower_cpu(
    tokens: torch.Tensor,
    lower: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Exact left-to-right merge preserving per-token lowercase_count."""
    if tokens.device.type != "cpu" or lower.device.type != "cpu":
        raise ValueError("CPU tensors expected")
    if tokens.numel() != lower.numel():
        raise ValueError("tokens/lower length mismatch")
    if tokens.numel() < 2:
        return tokens, lower

    matches = (tokens[:-1] == left) & (tokens[1:] == right)
    starts = torch.nonzero(matches, as_tuple=False).flatten()
    if starts.numel() == 0:
        return tokens, lower

    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor([True], dtype=torch.bool),
            starts[1:] != starts[:-1] + 1,
        ))
        run_ids = torch.cumsum(new_run.to(torch.int64), dim=0) - 1
        run_starts = torch.nonzero(new_run, as_tuple=False).flatten()
        offsets = torch.arange(starts.numel(), dtype=torch.int64) - run_starts[run_ids]
        starts = starts[offsets.remainder(2) == 0]

    out_tokens = tokens.clone()
    out_lower = lower.clone()
    out_tokens[starts] = new_token
    out_lower[starts] = lower[starts] + lower[starts + 1]

    remove = torch.zeros(tokens.numel(), dtype=torch.bool)
    remove[starts + 1] = True
    keep = ~remove

    return (
        out_tokens[keep].contiguous(),
        out_lower[keep].contiguous(),
    )


def apply_bpe_cpu(raw_cpu: torch.Tensor, *, reverse_complement: bool):
    raw = raw_cpu.to(dtype=torch.uint8, device="cpu").contiguous()
    if reverse_complement:
        raw = reverse_complement_storage_codes_cpu(raw)

    tokens = torch.where(
        raw < 8,
        torch.remainder(raw, 4),
        torch.full_like(raw, 4),
    ).to(torch.uint8).contiguous()
    lower = ((raw >= 4) & (raw < 8)).to(torch.uint8).contiguous()

    for left, right, new_token in merge_history:
        tokens, lower = _merge_tokens_and_lower_cpu(
            tokens, lower, left, right, new_token
        )

    return tokens, lower


def safe_name(contig: str):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", contig)[:80]
    suffix = hashlib.sha1(contig.encode()).hexdigest()[:10]
    return f"{stem}_{suffix}"


def cache_path(contig: str, orientation: str):
    return CACHE_DIR / f"{safe_name(contig)}.{orientation}.pt"


def _build_token_ends(token_ids: torch.Tensor):
    n_tokens = token_ids.numel()
    ends = torch.empty(n_tokens, dtype=torch.int32)
    running_bp = 0

    for start in range(0, n_tokens, INDEX_CHUNK_TOKENS):
        stop = min(start + INDEX_CHUNK_TOKENS, n_tokens)
        lengths = TOKEN_LENGTH_LOOKUP[token_ids[start:stop].long()].to(torch.int32)
        chunk_ends = torch.cumsum(lengths, dim=0) + running_bp
        ends[start:stop] = chunk_ends
        running_bp = int(chunk_ends[-1])

    return ends, running_bp


def cache_is_compatible(path: Path, contig: str, orientation: str):
    if not path.is_file():
        return False
    try:
        p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
        return (
            int(p.get("cache_format_version", -1)) == CACHE_FORMAT_VERSION
            and p.get("cache_engine") == CACHE_ENGINE
            and p.get("merge_hash") == MERGE_HASH
            and int(p.get("bpe_merges", -1)) == BPE_MERGES
            and p.get("contig") == contig
            and p.get("orientation") == orientation
            and "token_ids" in p
            and "lowercase_count" in p
            and "token_ends" in p
        )
    except Exception:
        return False


def pretokenize_contig_cpu(contig: str, orientation: str, overwrite=False):
    if orientation not in {"plus", "rc"}:
        raise ValueError(orientation)

    path = cache_path(contig, orientation)
    if not overwrite and cache_is_compatible(path, contig, orientation):
        print(f"{contig} [{orientation}] cached")
        return path

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8).contiguous()
    lowercase_bp = int(((raw >= 4) & (raw < 8)).sum())

    ids, lower = apply_bpe_cpu(
        raw,
        reverse_complement=(orientation == "rc"),
    )
    ends, reconstructed_bp = _build_token_ends(ids)

    assert reconstructed_bp == raw.numel()
    assert ids.numel() == lower.numel() == ends.numel()
    assert int(lower.to(torch.int64).sum()) == lowercase_bp
    assert int(ids.max()) < vocab_size if ids.numel() else True

    payload = {
        "cache_format_version": CACHE_FORMAT_VERSION,
        "cache_engine": CACHE_ENGINE,
        "contig": contig,
        "orientation": orientation,
        "length_bp": int(raw.numel()),
        "lowercase_bp": lowercase_bp,
        "token_ids": ids,
        "lowercase_count": lower,
        "token_ends": ends,
        "merge_hash": MERGE_HASH,
        "bpe_merges": BPE_MERGES,
        "vocab_size": vocab_size,
    }
    torch.save(payload, path)

    print(
        f"{contig} [{orientation}] bp={raw.numel():,} "
        f"tokens={ids.numel():,} compression={raw.numel() / ids.numel():.3f}x"
    )

    del raw, ids, lower, ends, payload
    gc.collect()
    return path


In [8]:
# ============================================================
# CPU-ONLY BPE SEMANTIC SMOKE TEST
# ============================================================
# Full-contig BPE is intentionally CPU-only.
# DirectML is not touched here.

smoke_contig = train_contigs[0]

smoke_raw = torch.as_tensor(
    sequences[smoke_contig][:GLOBAL_LENGTH],
    dtype=torch.uint8,
).contiguous()

if smoke_raw.numel() < GLOBAL_LENGTH:
    smoke_raw = torch.cat((
        smoke_raw,
        torch.full(
            (GLOBAL_LENGTH - smoke_raw.numel(),),
            8,
            dtype=torch.uint8,
        ),
    ))

for rc in (False, True):

    ids, lower = apply_bpe_cpu(
        smoke_raw,
        reverse_complement=rc,
    )

    lengths = TOKEN_LENGTH_LOOKUP[
        ids.long()
    ].to(torch.int32)

    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ids.numel() == lower.numel()
    assert int(lengths.sum()) == GLOBAL_LENGTH

    assert torch.all(
        lower.to(torch.int16)
        <= lengths.to(torch.int16)
    )

    source = (
        reverse_complement_storage_codes_cpu(smoke_raw)
        if rc
        else smoke_raw
    )

    expected_lowercase = int(
        (
            (source >= 4)
            & (source < 8)
        ).sum()
    )

    assert (
        int(lower.to(torch.int64).sum())
        == expected_lowercase
    )

    print(
        f"{'RC' if rc else 'PLUS'}: "
        f"tokens={ids.numel():,} "
        f"compression="
        f"{GLOBAL_LENGTH / ids.numel():.3f}x"
    )

print(
    f"BPE-{BPE_MERGES} CPU semantic check: OK"
)


PLUS: tokens=4,348 compression=3.297x
RC: tokens=4,360 compression=3.288x
BPE-200 CPU semantic check: OK


In [9]:
# ============================================================
# BUILD EXACT BPE-200 PRETOKENIZED CACHE — ONE TIME, RESUME-SAFE
# ============================================================
# Old DirectML cache is intentionally ignored because CACHE_DIR changed.

OVERWRITE_CACHE = False

started = time.perf_counter()
for i, contig in enumerate(cache_contigs, 1):
    print(f"\n[{i}/{len(cache_contigs)}] {contig}")
    pretokenize_contig_cpu(contig, "plus", OVERWRITE_CACHE)
    pretokenize_contig_cpu(contig, "rc", OVERWRITE_CACHE)

print()
print("Exact cache ready. Minutes:", round((time.perf_counter() - started) / 60, 2))



[1/12] NC_064035.1
NC_064035.1 [plus] cached
NC_064035.1 [rc] cached

[2/12] NC_064036.1
NC_064036.1 [plus] cached
NC_064036.1 [rc] cached

[3/12] NC_064038.1
NC_064038.1 [plus] cached
NC_064038.1 [rc] cached

[4/12] NC_064039.1
NC_064039.1 [plus] cached
NC_064039.1 [rc] cached

[5/12] NC_064040.1
NC_064040.1 [plus] cached
NC_064040.1 [rc] cached

[6/12] NC_064043.1
NC_064043.1 [plus] cached
NC_064043.1 [rc] cached

[7/12] NC_064045.1
NC_064045.1 [plus] cached
NC_064045.1 [rc] cached

[8/12] NC_064047.1
NC_064047.1 [plus] cached
NC_064047.1 [rc] cached

[9/12] NC_064048.1
NC_064048.1 [plus] cached
NC_064048.1 [rc] cached

[10/12] NC_064034.1
NC_064034.1 [plus] cached
NC_064034.1 [rc] cached

[11/12] NC_064041.1
NC_064041.1 [plus] cached
NC_064041.1 [rc] cached

[12/12] NC_064042.1
NC_064042.1 [plus] cached
NC_064042.1 [rc] cached

Exact cache ready. Minutes: 0.0


In [10]:
# ============================================================
# FULL CACHE AUDIT
# ============================================================
# Verifies metadata, bp reconstruction, lowercase counts and boundary index.

AUDIT_CHUNK_TOKENS = 1_000_000


def audit_cache_file(contig: str, orientation: str):
    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Incompatible cache: {path}")

    p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    ids = p["token_ids"]
    lower = p["lowercase_count"]
    ends = p["token_ends"]

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8)
    expected_bp = raw.numel()
    expected_lower = int(((raw >= 4) & (raw < 8)).sum())

    assert int(p["length_bp"]) == expected_bp
    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ends.dtype == torch.int32
    assert ids.numel() == lower.numel() == ends.numel()
    assert ids.numel() > 0
    assert int(ids.max()) < vocab_size

    running_bp = 0
    lower_total = 0

    for start in range(0, ids.numel(), AUDIT_CHUNK_TOKENS):
        stop = min(start + AUDIT_CHUNK_TOKENS, ids.numel())
        chunk_ids = ids[start:stop]
        chunk_lower = lower[start:stop]
        chunk_ends = ends[start:stop]
        lengths = TOKEN_LENGTH_LOOKUP[chunk_ids.long()].to(torch.int32)
        expected_ends = torch.cumsum(lengths, dim=0) + running_bp

        assert torch.equal(chunk_ends, expected_ends)
        assert torch.all(chunk_lower.to(torch.int16) <= lengths.to(torch.int16))

        running_bp = int(expected_ends[-1])
        lower_total += int(chunk_lower.to(torch.int64).sum())

    assert running_bp == expected_bp
    assert lower_total == expected_lower == int(p["lowercase_bp"])

    return {
        "contig": contig,
        "orientation": orientation,
        "bp": expected_bp,
        "tokens": ids.numel(),
        "compression": expected_bp / ids.numel(),
        "lowercase_bp": lower_total,
    }


audit_rows = []
started = time.perf_counter()

for i, contig in enumerate(cache_contigs, 1):
    print(f"[{i}/{len(cache_contigs)}] {contig}")
    for orientation in ("plus", "rc"):
        row = audit_cache_file(contig, orientation)
        audit_rows.append(row)
        print(
            f"  {orientation:4s} tokens={row['tokens']:,} "
            f"compression={row['compression']:.3f}x"
        )

cache_audit = pd.DataFrame(audit_rows)
display(cache_audit)
print("Audit minutes:", round((time.perf_counter() - started) / 60, 2))
print("CACHE AUDIT: OK")


[1/12] NC_064035.1
  plus tokens=6,575,767 compression=3.350x
  rc   tokens=6,572,480 compression=3.352x
[2/12] NC_064036.1
  plus tokens=6,214,416 compression=3.346x
  rc   tokens=6,211,319 compression=3.348x
[3/12] NC_064038.1
  plus tokens=5,669,670 compression=3.351x
  rc   tokens=5,674,624 compression=3.348x
[4/12] NC_064039.1
  plus tokens=5,406,734 compression=3.347x
  rc   tokens=5,407,155 compression=3.347x
[5/12] NC_064040.1
  plus tokens=5,323,448 compression=3.356x
  rc   tokens=5,321,541 compression=3.357x
[6/12] NC_064043.1
  plus tokens=4,943,267 compression=3.357x
  rc   tokens=4,953,485 compression=3.350x
[7/12] NC_064045.1
  plus tokens=4,889,799 compression=3.352x
  rc   tokens=4,891,140 compression=3.351x
[8/12] NC_064047.1
  plus tokens=4,596,057 compression=3.345x
  rc   tokens=4,593,831 compression=3.346x
[9/12] NC_064048.1
  plus tokens=4,074,317 compression=3.367x
  rc   tokens=4,073,645 compression=3.367x
[10/12] NC_064034.1
  plus tokens=6,627,064 compression

,contig,orientation,bp,tokens,compression,lowercase_bp
0,NC_064035.1,plus,22031953,6575767,3.350477,5303374
1,NC_064035.1,rc,22031953,6572480,3.352152,5303374
2,NC_064036.1,plus,20793841,6214416,3.346065,5356286
3,NC_064036.1,rc,20793841,6211319,3.347734,5356286
4,NC_064038.1,plus,19000648,5669670,3.351279,7404189
5,NC_064038.1,rc,19000648,5674624,3.348354,7404189
6,NC_064039.1,plus,18096078,5406734,3.346952,4604642
7,NC_064039.1,rc,18096078,5407155,3.346691,4604642
8,NC_064040.1,plus,17865913,5323448,3.356079,5523134
9,NC_064040.1,rc,17865913,5321541,3.357282,5523134


Audit minutes: 0.02
CACHE AUDIT: OK


In [11]:
# ============================================================
# LAZY MMAP CACHE
# ============================================================
# These are mmap handles, not full copies of all tensors in RAM.
# Keeping all 24 orientation files mapped avoids repeated torch.load in training.

CACHE_MAX_ITEMS = max(32, 2 * len(cache_contigs))
_CACHE_LRU = OrderedDict()


def load_cached_lazy(contig: str, orientation: str):
    key = (contig, orientation)

    if key in _CACHE_LRU:
        item = _CACHE_LRU.pop(key)
        _CACHE_LRU[key] = item
        return item

    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Missing/incompatible cache: {path}")

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

    item = {
        "payload": payload,  # keeps mmap storage alive
        "length_bp": int(payload["length_bp"]),
        "ids": payload["token_ids"],
        "lower": payload["lowercase_count"],
        "ends": payload["token_ends"],
    }

    _CACHE_LRU[key] = item
    while len(_CACHE_LRU) > CACHE_MAX_ITEMS:
        _CACHE_LRU.popitem(last=False)
        gc.collect()

    return item


print("Lazy mmap cache ready.")
print("Mapped files now:", len(_CACHE_LRU))
print("No full-contig lengths/cumsum tensors are created in RAM.")


Lazy mmap cache ready.
Mapped files now: 0
No full-contig lengths/cumsum tensors are created in RAM.


In [12]:
# ============================================================
# FAST WINDOW SLICING — NO BPE MERGES HERE
# ============================================================

ATTN_CONTEXT_STRIDE = 16
ATTN_CONTEXT_OFFSET = ATTN_CONTEXT_STRIDE // 2

CONTEXT_POSITIONS_BP = torch.arange(
    ATTN_CONTEXT_OFFSET,
    GLOBAL_LENGTH,
    ATTN_CONTEXT_STRIDE,
    dtype=torch.float32,
)

N_CONTEXT_POSITIONS = int(
    CONTEXT_POSITIONS_BP.numel()
)

assert GLOBAL_LENGTH % ATTN_CONTEXT_STRIDE == 0
assert N_CONTEXT_POSITIONS == GLOBAL_LENGTH // ATTN_CONTEXT_STRIDE



def _search_token_index(ends: torch.Tensor, bp_coordinate: int):
    query = torch.tensor(bp_coordinate, dtype=ends.dtype)
    return int(torch.searchsorted(ends, query, right=True))


def slice_cached(contig, genomic_start, genomic_end, *, rc=False):
    orientation = "rc" if rc else "plus"
    item = load_cached_lazy(contig, orientation)
    L = item["length_bp"]

    if rc:
        start, end = L - genomic_end, L - genomic_start
    else:
        start, end = genomic_start, genomic_end

    if end - start != GLOBAL_LENGTH:
        raise AssertionError((start, end, GLOBAL_LENGTH))

    left_pad = max(0, -start)
    right_pad = max(0, end - L)
    a, b = max(0, start), min(L, end)

    ids_all = item["ids"]
    lower_all = item["lower"]
    ends_all = item["ends"]

    if b > a:
        first = _search_token_index(ends_all, a)
        last = _search_token_index(ends_all, b - 1)
        stop = last + 1

        core_ids = ids_all[first:stop]
        core_lower = lower_all[first:stop]
        core_lengths = TOKEN_LENGTH_LOOKUP[core_ids.long()].to(torch.int16)
        core_ends_abs = ends_all[first:stop]

        # Search only the small token slice, not the full 20M-element index.
        coords = torch.arange(a, b, dtype=core_ends_abs.dtype)
        core_map = (
            torch.searchsorted(core_ends_abs, coords, right=True)
            + left_pad
        ).to(torch.long)
    else:
        core_ids = torch.empty(0, dtype=torch.uint8)
        core_lower = torch.empty(0, dtype=torch.uint8)
        core_lengths = torch.empty(0, dtype=torch.int16)
        core_map = torch.empty(0, dtype=torch.long)

    # N is intentionally never BPE-merged, so 1 padded bp = 1 N token.
    left_ids = torch.full((left_pad,), N_TOKEN_ID, dtype=torch.uint8)
    left_lower = torch.zeros(left_pad, dtype=torch.uint8)
    left_lengths = torch.ones(left_pad, dtype=torch.int16)
    left_map = torch.arange(left_pad, dtype=torch.long)

    right_ids = torch.full((right_pad,), N_TOKEN_ID, dtype=torch.uint8)
    right_lower = torch.zeros(right_pad, dtype=torch.uint8)
    right_lengths = torch.ones(right_pad, dtype=torch.int16)
    right_map = (
        left_pad
        + core_ids.numel()
        + torch.arange(right_pad, dtype=torch.long)
    )

    ids = torch.cat((left_ids, core_ids, right_ids)).contiguous()
    lower = torch.cat((left_lower, core_lower, right_lower)).contiguous()
    lengths = torch.cat((left_lengths, core_lengths, right_lengths)).contiguous()
    bp_to_token = torch.cat((left_map, core_map, right_map)).contiguous()

    assert bp_to_token.numel() == GLOBAL_LENGTH
    assert ids.numel() == lengths.numel() == lower.numel()
    assert bp_to_token.min() >= 0
    assert bp_to_token.max() < ids.numel()

    bp_valid = torch.cat(
        (
            torch.zeros(
                left_pad,
                dtype=torch.float32,
            ),
            torch.ones(
                b - a,
                dtype=torch.float32,
            ),
            torch.zeros(
                right_pad,
                dtype=torch.float32,
            ),
        )
    ).contiguous()

    assert bp_valid.numel() == GLOBAL_LENGTH

    return {
        "ids": ids,
        "lengths": lengths,
        "softmask": lower.float() / lengths.float(),
        "bp_to_token": bp_to_token,
        "bp_valid": bp_valid,
    }


def build_cached_both_batch(tiles):
    items = []

    # Ordering intentionally matches batch.x_both_strands:
    # all PLUS rows first, then all RC rows.
    for rc in (False, True):
        for i in range(len(tiles)):
            tile = tiles.iloc[i]
            item = slice_cached(
                str(tile["contig"]),
                int(tile["target_start"]) - GLOBAL_FLANK,
                int(tile["target_end"]) + GLOBAL_FLANK,
                rc=rc,
            )
            item["local_map"] = item["bp_to_token"][
                LOCAL_START_IN_GLOBAL:LOCAL_END_IN_GLOBAL
            ]

            item["context_map"] = item["bp_to_token"][
                ATTN_CONTEXT_OFFSET:
                GLOBAL_LENGTH:
                ATTN_CONTEXT_STRIDE
            ]

            item["context_valid"] = item["bp_valid"][
                ATTN_CONTEXT_OFFSET:
                GLOBAL_LENGTH:
                ATTN_CONTEXT_STRIDE
            ]

            assert item["local_map"].numel() == LOCAL_LENGTH
            assert item["context_map"].numel() == N_CONTEXT_POSITIONS
            assert item["context_valid"].numel() == N_CONTEXT_POSITIONS

            items.append(item)

    T = max(x["ids"].numel() for x in items)
    N = len(items)

    token_ids = torch.full((N, T), PAD_TOKEN_ID, dtype=torch.long)
    token_lengths = torch.zeros((N, T), dtype=torch.float32)
    softmask = torch.zeros((N, T), dtype=torch.float32)
    token_mask = torch.zeros((N, T), dtype=torch.float32)  # float for DirectML
    local_map = torch.empty((N, LOCAL_LENGTH), dtype=torch.long)
    context_map = torch.empty(
        (N, N_CONTEXT_POSITIONS),
        dtype=torch.long,
    )
    context_valid = torch.empty(
        (N, N_CONTEXT_POSITIONS),
        dtype=torch.float32,
    )

    for row, x in enumerate(items):
        n = x["ids"].numel()
        token_ids[row, :n] = x["ids"].long()
        token_lengths[row, :n] = x["lengths"].float()
        softmask[row, :n] = x["softmask"]
        token_mask[row, :n] = 1.0
        local_map[row] = x["local_map"]
        context_map[row] = x["context_map"]
        context_valid[row] = x["context_valid"]

    return {
        "token_ids": token_ids.contiguous(),
        "token_lengths": token_lengths.contiguous(),
        "softmask": softmask.contiguous(),
        "token_mask": token_mask.contiguous(),
        "local_map": local_map.contiguous(),
        "context_map": context_map.contiguous(),
        "context_valid": context_valid.contiguous(),
    }


# Real batch smoke-test.
smoke_batch = train_generator.sample_batch(batch_size=4)
t0 = time.perf_counter()
smoke_global = build_cached_both_batch(smoke_batch.tiles)
cache_build_ms = (time.perf_counter() - t0) * 1000

print("Cached batch build ms:", round(cache_build_ms, 2))
print("Token batch:", smoke_global["token_ids"].shape)
print("Mapped files:", len(_CACHE_LRU))
assert smoke_global["token_ids"].shape[0] == 8
assert smoke_global["local_map"].shape == (8, LOCAL_LENGTH)
assert smoke_global["context_map"].shape == (
    8,
    N_CONTEXT_POSITIONS,
)
assert smoke_global["context_valid"].shape == (
    8,
    N_CONTEXT_POSITIONS,
)
print(
    "Context anchors:",
    N_CONTEXT_POSITIONS,
    "stride:",
    ATTN_CONTEXT_STRIDE,
    "bp",
)
print("Cached slicing smoke-test: OK")


# ============================================================
# BPE TOKEN-LENGTH vs GLOBAL CNN RF DIAGNOSTIC
# ============================================================
# Current Global encoder theoretical receptive field:
RF_LIMIT_TOKENS = 4093

probe_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=4242,
)

probe_token_counts = []

RF_PROBE_BATCHES = 8
RF_PROBE_BATCH_SIZE = 4

for _ in range(RF_PROBE_BATCHES):
    probe_batch = probe_generator.sample_batch(
        batch_size=RF_PROBE_BATCH_SIZE
    )

    probe_global = build_cached_both_batch(
        probe_batch.tiles
    )

    counts = (
        probe_global["token_mask"]
        .sum(dim=1)
        .cpu()
        .numpy()
    )

    probe_token_counts.extend(
        counts.tolist()
    )

probe_token_counts = pd.Series(
    probe_token_counts,
    dtype="float64",
)

print()
print("=== BPE TOKENS PER 14,336-bp GLOBAL WINDOW ===")
print("samples:", len(probe_token_counts))
print("min:", int(probe_token_counts.min()))
print("median:", round(float(probe_token_counts.median()), 1))
print("p90:", round(float(probe_token_counts.quantile(0.90)), 1))
print("p95:", round(float(probe_token_counts.quantile(0.95)), 1))
print("max:", int(probe_token_counts.max()))

fraction_within_rf = float(
    (probe_token_counts <= RF_LIMIT_TOKENS).mean()
)

print(
    "fraction <= RF(4093 tokens):",
    f"{fraction_within_rf:.3f}",
)

print(
    "fraction > RF(4093 tokens):",
    f"{1.0 - fraction_within_rf:.3f}",
)

if fraction_within_rf < 0.90:
    print(
        "WARNING: many windows exceed the current token RF. "
        "If AP stops improving, expand Global CNN RF before adding more context."
    )
else:
    print(
        "RF diagnostic: most sampled windows fit inside the current token RF."
    )


Cached batch build ms: 10.81
Token batch: torch.Size([8, 4360])
Mapped files: 6
Context anchors: 896 stride: 16 bp
Cached slicing smoke-test: OK

=== BPE TOKENS PER 14,336-bp GLOBAL WINDOW ===
samples: 64
min: 4173
median: 4273.5
p90: 4363.7
p95: 4381.2
max: 4408
fraction <= RF(4093 tokens): 0.000
fraction > RF(4093 tokens): 1.000


In [13]:
# ============================================================
# MODEL — STRONG EXP020 CNN + DYNAMIC CROSS-ATTENTION
# 32-BP HEAD ONLY
# ============================================================

GLOBAL_DILATIONS = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)

GLOBAL_TOKEN_RF = (
    1
    + sum(
        4 * d
        for d in GLOBAL_DILATIONS
    )
)

assert GLOBAL_TOKEN_RF == 4093
assert GLOBAL_TOKEN_RF == RF_LIMIT_TOKENS

REGION_WIDTHS = (
    32,
)

REGION_WIDTH = 32
N_REGIONS = TARGET_LENGTH // REGION_WIDTH

assert N_REGIONS == 64

GLOBAL_CHANNELS = 32

ATTN_DIM = 64
ATTN_HEADS = 4
ATTN_HEAD_DIM = ATTN_DIM // ATTN_HEADS
ATTN_SCALE = ATTN_HEAD_DIM ** -0.5
ATTN_GATE_INIT = 0.10

# ------------------------------------------------------------
# SIGNED RELATIVE-POSITION BIAS
# ------------------------------------------------------------

REL_BIAS_BIN_BP = 128
REL_BIAS_MAX_BP = 6400

REL_BIAS_SIDE_BINS = int(
    math.ceil(
        REL_BIAS_MAX_BP
        / REL_BIAS_BIN_BP
    )
)

REL_BIAS_CENTER = (
    REL_BIAS_SIDE_BINS
)

REL_BIAS_BUCKETS = (
    2 * REL_BIAS_SIDE_BINS
    + 1
)

assert REL_BIAS_SIDE_BINS == 50
assert REL_BIAS_BUCKETS == 101
assert ATTN_DIM % ATTN_HEADS == 0


QUERY_CENTERS_BP = (
    GLOBAL_FLANK
    + (
        torch.arange(
            N_REGIONS,
            dtype=torch.float32,
        )
        + 0.5
    )
    * REGION_WIDTH
)

RELATIVE_DELTA_BP = (
    CONTEXT_POSITIONS_BP[
        None,
        :
    ]
    - QUERY_CENTERS_BP[
        :,
        None,
    ]
)


def make_signed_relative_bucket_ids(
    delta_bp,
):
    abs_delta = torch.abs(
        delta_bp
    )

    magnitude = torch.ceil(
        abs_delta
        / float(
            REL_BIAS_BIN_BP
        )
    ).to(
        torch.long
    )

    magnitude = torch.clamp(
        magnitude,
        min=1,
        max=REL_BIAS_SIDE_BINS,
    )

    center = torch.full_like(
        magnitude,
        REL_BIAS_CENTER,
    )

    bucket = torch.where(
        delta_bp < 0,
        center - magnitude,
        torch.where(
            delta_bp > 0,
            center + magnitude,
            center,
        ),
    )

    return bucket.contiguous()


RELATIVE_BUCKET_IDS = (
    make_signed_relative_bucket_ids(
        RELATIVE_DELTA_BP
    )
)

assert RELATIVE_BUCKET_IDS.shape == (
    N_REGIONS,
    N_CONTEXT_POSITIONS,
)

assert int(
    RELATIVE_BUCKET_IDS.min()
) >= 0

assert int(
    RELATIVE_BUCKET_IDS.max()
) < REL_BIAS_BUCKETS


class GlobalResidualBlock(nn.Module):
    def __init__(
        self,
        channels,
        dilation,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.act = nn.GELU()

    def forward(
        self,
        x,
        mask,
    ):
        m = (
            mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )

        residual = x

        x = (
            self.act(
                self.conv1(x)
            )
            * m
        )

        return (
            self.act(
                self.conv2(x)
                + residual
            )
            * m
        )


class GlobalBPEEncoder(nn.Module):
    """
    Strong path is exactly the EXP020 dilated CNN.

    We also return the shallow stem feature map as the attention
    K/V source. This makes attention distance interpretable:
    an attended anchor is still a local BPE representation,
    not an already-nearly-global RF=4093 feature.
    """
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size + 1,
            16,
            padding_idx=PAD_TOKEN_ID,
        )

        self.proj = nn.Sequential(
            nn.Conv1d(
                18,
                GLOBAL_CHANNELS,
                1,
            ),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList(
            GlobalResidualBlock(
                GLOBAL_CHANNELS,
                d,
            )
            for d in GLOBAL_DILATIONS
        )

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
    ):
        emb = self.embedding(ids)

        length_feature = (
            lengths.to(emb.dtype)
            / float(MAX_TOKEN_LENGTH)
        ).unsqueeze(-1)

        soft_feature = (
            softmask.to(emb.dtype)
            .unsqueeze(-1)
        )

        x = torch.cat(
            (
                emb,
                soft_feature,
                length_feature,
            ),
            dim=-1,
        )

        x = (
            x
            * mask.unsqueeze(-1).to(
                dtype=x.dtype
            )
        )

        stem = self.proj(
            x.transpose(
                1,
                2,
            ).contiguous()
        )

        stem = (
            stem
            * mask.unsqueeze(1).to(
                dtype=stem.dtype
            )
        )

        strong = stem

        for block in self.blocks:
            strong = block(
                strong,
                mask,
            )

        return (
            strong,
            stem,
        )


def pool_regions_32(
    bp_features,
):
    B, C, L = bp_features.shape

    if L != TARGET_LENGTH:
        raise ValueError(
            (L, TARGET_LENGTH)
        )

    x = bp_features.reshape(
        B,
        C,
        N_REGIONS,
        REGION_WIDTH,
    )

    mean_features = x.mean(
        dim=-1
    )

    rms_features = torch.sqrt(
        (
            x * x
        ).mean(
            dim=-1
        )
        + 1e-6
    )

    return torch.cat(
        (
            mean_features,
            rms_features,
        ),
        dim=1,
    ).contiguous()


class DirectMLCrossAttention(nn.Module):
    """
    Custom multi-head cross-attention with learned signed relative-position bias.

    Q: one query per 32-bp target region.
    K/V: regularly spaced global bp anchors.

    Avoids nn.MultiheadAttention / SDPA fused paths so the primitive
    operations remain explicit for DirectML.
    """
    def __init__(self):
        super().__init__()

        self.q_proj = nn.Conv1d(
            2 * GLOBAL_CHANNELS,
            ATTN_DIM,
            1,
        )

        self.k_proj = nn.Conv1d(
            GLOBAL_CHANNELS,
            ATTN_DIM,
            1,
        )

        self.v_proj = nn.Conv1d(
            GLOBAL_CHANNELS,
            ATTN_DIM,
            1,
        )

        self.out_proj = nn.Conv1d(
            ATTN_DIM,
            ATTN_DIM,
            1,
        )

        # One learnable signed relative-distance curve per head.
        # Shape after lookup: [R, S, H] -> [1, H, R, S].
        self.relative_bias = nn.Embedding(
            REL_BIAS_BUCKETS,
            ATTN_HEADS,
        )

        nn.init.zeros_(
            self.relative_bias.weight
        )

        self.register_buffer(
            "relative_bucket_ids",
            RELATIVE_BUCKET_IDS.clone(),
            persistent=True,
        )

    def forward(
        self,
        region_features,
        context_features,
        context_valid,
        *,
        return_attention=False,
    ):
        B = region_features.shape[0]
        R = region_features.shape[-1]
        S = context_features.shape[-1]

        q = (
            self.q_proj(
                region_features
            )
            .reshape(
                B,
                ATTN_HEADS,
                ATTN_HEAD_DIM,
                R,
            )
            .permute(
                0,
                1,
                3,
                2,
            )
            .contiguous()
        )

        k = (
            self.k_proj(
                context_features
            )
            .reshape(
                B,
                ATTN_HEADS,
                ATTN_HEAD_DIM,
                S,
            )
            .permute(
                0,
                1,
                3,
                2,
            )
            .contiguous()
        )

        v = (
            self.v_proj(
                context_features
            )
            .reshape(
                B,
                ATTN_HEADS,
                ATTN_HEAD_DIM,
                S,
            )
            .permute(
                0,
                1,
                3,
                2,
            )
            .contiguous()
        )

        scores = (
            torch.matmul(
                q,
                k.transpose(
                    -1,
                    -2,
                ).contiguous(),
            )
            * ATTN_SCALE
        )

        relative_bias = (
            self.relative_bias(
                self.relative_bucket_ids
            )
            .permute(
                2,
                0,
                1,
            )
            .unsqueeze(
                0
            )
            .to(
                dtype=scores.dtype
            )
        )

        scores = (
            scores
            + relative_bias
        )

        valid = (
            context_valid
            .reshape(
                B,
                1,
                1,
                S,
            )
            .to(
                dtype=scores.dtype
            )
        )

        # Additive mask is DirectML-friendlier than advanced boolean indexing.
        scores = (
            scores
            + (
                1.0 - valid
            ) * (-1.0e4)
        )

        weights = torch.softmax(
            scores,
            dim=-1,
        )

        attended = torch.matmul(
            weights,
            v,
        )

        attended = (
            attended
            .permute(
                0,
                1,
                3,
                2,
            )
            .contiguous()
            .reshape(
                B,
                ATTN_DIM,
                R,
            )
        )

        attended = self.out_proj(
            attended
        )

        if return_attention:
            return (
                attended,
                weights,
            )

        return (
            attended,
            None,
        )


class CrossAttentionRegionalHead(nn.Module):
    """
    Keep the original regional classifier as an intact baseline path.

    Attention contributes only through a gated residual logit delta:
        final = base + gate * delta

    This lets attention prove whether it helps instead of forcing it.
    """
    def __init__(self):
        super().__init__()

        self.attention = (
            DirectMLCrossAttention()
        )

        self.base_classifier = nn.Sequential(
            nn.Conv1d(
                2 * GLOBAL_CHANNELS,
                32,
                1,
            ),
            nn.GELU(),
            nn.Conv1d(
                32,
                1,
                1,
            ),
        )

        self.attn_classifier = nn.Sequential(
            nn.Conv1d(
                ATTN_DIM,
                32,
                1,
            ),
            nn.GELU(),
            nn.Conv1d(
                32,
                1,
                1,
            ),
        )

        self.attn_gate = nn.Parameter(
            torch.tensor(
                ATTN_GATE_INIT,
                dtype=torch.float32,
            )
        )

    def forward(
        self,
        region_features,
        context_features,
        context_valid,
        *,
        return_attention=False,
    ):
        base_logits = (
            self.base_classifier(
                region_features
            )
        )

        (
            attended,
            weights,
        ) = self.attention(
            region_features,
            context_features,
            context_valid,
            return_attention=return_attention,
        )

        delta_logits = (
            self.attn_classifier(
                attended
            )
        )

        gate = (
            self.attn_gate
            .reshape(
                1,
                1,
                1,
            )
            .to(
                dtype=delta_logits.dtype
            )
        )

        logits = (
            base_logits
            + gate * delta_logits
        )

        return (
            logits,
            weights,
        )


class GlobalRegionalBPE200CrossAttentionModel(
    nn.Module
):
    def __init__(self):
        super().__init__()

        self.global_encoder = (
            GlobalBPEEncoder()
        )

        self.regional_head = (
            CrossAttentionRegionalHead()
        )

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
        local_map,
        context_map,
        context_valid,
        *,
        return_attention=False,
    ):
        (
            strong_tokens,
            stem_tokens,
        ) = self.global_encoder(
            ids,
            lengths,
            softmask,
            mask,
        )

        # --------------------------------------------------------
        # 32-bp region queries from the strong EXP020 representation
        # --------------------------------------------------------
        local_gather = (
            local_map
            .unsqueeze(1)
            .expand(
                -1,
                strong_tokens.shape[1],
                -1,
            )
            .contiguous()
        )

        bp_features = torch.gather(
            strong_tokens,
            2,
            local_gather,
        )

        bp_features = bp_features[
            :,
            :,
            LOCAL_FLANK:-LOCAL_FLANK,
        ]

        region_features = (
            pool_regions_32(
                bp_features
            )
        )

        # --------------------------------------------------------
        # Global K/V bank on a regular 16-bp genomic grid.
        # Source = shallow stem, intentionally local/interpretable.
        # --------------------------------------------------------
        context_gather = (
            context_map
            .unsqueeze(1)
            .expand(
                -1,
                stem_tokens.shape[1],
                -1,
            )
            .contiguous()
        )

        context_features = torch.gather(
            stem_tokens,
            2,
            context_gather,
        )

        (
            logits,
            attention_weights,
        ) = self.regional_head(
            region_features,
            context_features,
            context_valid,
            return_attention=return_attention,
        )

        outputs = {
            32: logits,
        }

        if return_attention:
            return (
                outputs,
                attention_weights,
            )

        return outputs


def create_model():
    return (
        GlobalRegionalBPE200CrossAttentionModel()
    )


def _move_global_batch_to_device(
    global_batch,
):
    return {
        key: value.contiguous().to(
            device
        )
        for key, value
        in global_batch.items()
    }


def forward_both(
    model,
    global_batch,
    *,
    grad=True,
):
    B2 = global_batch[
        "token_ids"
    ].shape[0]

    if B2 % 2 != 0:
        raise ValueError(
            f"Expected PLUS+RC rows, got {B2}"
        )

    B = B2 // 2

    g = _move_global_batch_to_device(
        global_batch
    )

    context = (
        torch.enable_grad()
        if grad
        else torch.no_grad()
    )

    with context:
        raw = model(
            g["token_ids"],
            g["token_lengths"],
            g["softmask"],
            g["token_mask"],
            g["local_map"],
            g["context_map"],
            g["context_valid"],
            return_attention=False,
        )

        outputs = {
            width: (
                logits[:B],
                logits[B:],
            )
            for width, logits
            in raw.items()
        }

    return outputs


def forward_both_with_attention(
    model,
    global_batch,
):
    """
    Evaluation-only helper.

    Returns:
      outputs:
        {32: (plus_logits, minus_reverse_logits)}
      attention:
        full oriented rows [2B, heads, 64 regions, context_positions]
    """
    B2 = global_batch[
        "token_ids"
    ].shape[0]

    if B2 % 2 != 0:
        raise ValueError(
            f"Expected PLUS+RC rows, got {B2}"
        )

    B = B2 // 2

    g = _move_global_batch_to_device(
        global_batch
    )

    with torch.no_grad():
        (
            raw,
            attention,
        ) = model(
            g["token_ids"],
            g["token_lengths"],
            g["softmask"],
            g["token_mask"],
            g["local_map"],
            g["context_map"],
            g["context_valid"],
            return_attention=True,
        )

    outputs = {
        width: (
            logits[:B],
            logits[B:],
        )
        for width, logits
        in raw.items()
    }

    return (
        outputs,
        attention,
    )


def make_region_targets_cpu(
    exact_target,
    base_weight,
    width,
):
    if exact_target.device.type != "cpu":
        raise ValueError(
            "Expected CPU exact_target"
        )

    if base_weight.device.type != "cpu":
        raise ValueError(
            "Expected CPU base_weight"
        )

    B = exact_target.shape[0]
    n_regions = TARGET_LENGTH // width

    target = (
        exact_target
        .float()
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .amax(
            dim=-1
        )
    )

    valid = (
        (
            base_weight > 0
        )
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .all(
            dim=-1
        )
        .float()
    )

    return (
        target.contiguous(),
        valid.contiguous(),
    )


def stable_region_bce(
    logits,
    targets,
    valid,
    *,
    max_pos_weight=32.0,
):
    targets = targets.to(
        dtype=logits.dtype
    )

    valid = valid.to(
        dtype=logits.dtype
    )

    with torch.no_grad():
        pos = (
            targets * valid
        ).sum()

        neg = (
            (1.0 - targets)
            * valid
        ).sum()

        pos_weight = torch.clamp(
            neg / torch.clamp(
                pos,
                min=1.0,
            ),
            min=1.0,
            max=max_pos_weight,
        )

    # Stable BCE-with-logits, avoiding unsupported DirectML paths.
    positive_term = (
        torch.clamp(
            -logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(logits)
            )
        )
    )

    negative_term = (
        torch.clamp(
            logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(logits)
            )
        )
    )

    element_loss = (
        targets
        * pos_weight
        * positive_term
        + (
            1.0 - targets
        )
        * negative_term
    )

    denom = torch.clamp(
        valid.sum(),
        min=1.0,
    )

    loss = (
        element_loss
        * valid
    ).sum() / denom

    return (
        loss,
        float(
            pos_weight.detach()
            .cpu()
            .item()
        ),
    )


# ============================================================
# ARCHITECTURE / SHAPE SMOKE TEST
# ============================================================

model = create_model().to(device)

encoder_parameters = sum(
    p.numel()
    for p
    in model.global_encoder.parameters()
)

attention_parameters = sum(
    p.numel()
    for p
    in model.regional_head.attention.parameters()
)

head_parameters = sum(
    p.numel()
    for p
    in model.regional_head.parameters()
)

total_parameters = sum(
    p.numel()
    for p
    in model.parameters()
)

print(
    "Architecture:",
    "EXP020 strong CNN + signed positional cross-attention",
)

print(
    "Global token RF:",
    GLOBAL_TOKEN_RF,
)

print(
    "Context stride:",
    ATTN_CONTEXT_STRIDE,
    "bp",
)

print(
    "Context anchors:",
    N_CONTEXT_POSITIONS,
)

print(
    "Queries:",
    N_REGIONS,
)

print(
    "Attention heads:",
    ATTN_HEADS,
)

print(
    "Attention dim:",
    ATTN_DIM,
)

print(
    "Relative-position bins:",
    REL_BIAS_BUCKETS,
    "x",
    REL_BIAS_BIN_BP,
    "bp signed",
)

print(
    "Relative-position max:",
    REL_BIAS_MAX_BP,
    "bp",
)

print(
    "Initial attention gate:",
    float(
        model.regional_head
        .attn_gate
        .detach()
        .cpu()
    ),
)

print(
    "Encoder parameters:",
    f"{encoder_parameters:,}",
)

print(
    "Attention parameters:",
    f"{attention_parameters:,}",
)

print(
    "Regional head parameters:",
    f"{head_parameters:,}",
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

model.eval()

smoke_outputs = forward_both(
    model,
    smoke_global,
    grad=False,
)

pp, mp = smoke_outputs[32]

assert pp.shape == (
    4,
    1,
    N_REGIONS,
)

assert mp.shape == (
    4,
    1,
    N_REGIONS,
)

print(
    "Output shape:",
    pp.shape,
)

# Attention-return smoke test on a smaller row count if possible.
small_smoke = {
    key: value[:2].contiguous()
    for key, value
    in smoke_global.items()
}

(
    small_outputs,
    small_attention,
) = forward_both_with_attention(
    model,
    small_smoke,
)

assert small_attention.shape == (
    2,
    ATTN_HEADS,
    N_REGIONS,
    N_CONTEXT_POSITIONS,
)

print(
    "Attention tensor:",
    small_attention.shape,
)

print(
    "Cross-attention smoke-test: OK"
)


Architecture: EXP020 strong CNN + signed positional cross-attention
Global token RF: 4093
Context stride: 16 bp
Context anchors: 896
Queries: 64
Attention heads: 4
Attention dim: 64
Relative-position bins: 101 x 128 bp signed
Relative-position max: 6400 bp
Initial attention gate: 0.10000000149011612
Encoder parameters: 65,984
Attention parameters: 12,948
Regional head parameters: 17,175
Total parameters: 83,159
Output shape: torch.Size([4, 1, 64])
Attention tensor: torch.Size([2, 4, 64, 896])
Cross-attention smoke-test: OK


In [14]:
# ============================================================
# ONE REAL BACKWARD STEP — DIRECTML SAFETY TEST
# ============================================================

torch.manual_seed(123)

smoke_model = (
    create_model()
    .to(device)
)

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=1e-3,
)

smoke_model.train()

smoke_optimizer.zero_grad(
    set_to_none=True
)

outputs = forward_both(
    smoke_model,
    smoke_global,
    grad=True,
)

pp, mp = outputs[32]

plus_target_r, plus_valid_r = (
    make_region_targets_cpu(
        smoke_batch.plus_target,
        smoke_batch.plus_loss_weight,
        32,
    )
)

minus_target_r, minus_valid_r = (
    make_region_targets_cpu(
        smoke_batch.minus_target_reverse,
        smoke_batch.minus_loss_weight_reverse,
        32,
    )
)

plus_loss, plus_pw = (
    stable_region_bce(
        pp,
        plus_target_r.to(device),
        plus_valid_r.to(device),
    )
)

minus_loss, minus_pw = (
    stable_region_bce(
        mp,
        minus_target_r.to(device),
        minus_valid_r.to(device),
    )
)

smoke_loss = (
    plus_loss
    + minus_loss
) / 2.0

smoke_loss.backward()
smoke_optimizer.step()

print(
    "Smoke loss:",
    float(
        smoke_loss.detach()
        .cpu()
    ),
)

print(
    "pos_weight:",
    round(
        (
            plus_pw
            + minus_pw
        )
        / 2.0,
        3,
    ),
)

print(
    "Attention gate after one step:",
    float(
        smoke_model.regional_head
        .attn_gate
        .detach()
        .cpu()
    ),
)

print(
    "DirectML backward smoke-test: OK"
)

del smoke_model
del smoke_optimizer
gc.collect()


Smoke loss: 1.0383753776550293
pos_weight: 28.125
Attention gate after one step: 0.0989999994635582
DirectML backward smoke-test: OK


15

In [15]:
# ============================================================
# EXP029 — 10 000-STEP POSITIONAL CROSS-ATTENTION SCREEN
# CROSS-ATTENTION + 32-BP HEAD ONLY
# SAFETY CHECKPOINT EVERY 2.5K
# ============================================================

STEPS = 10_000
SAVE_EVERY = 2_500
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42

MILESTONE_STEPS = (
    10_000,
)

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = create_model().to(device)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
)

model.train()

history = []

ema32 = None
seen = 0
prep_seconds = 0.0

started = time.perf_counter()


def checkpoint_path(step):
    return (
        RUN_DIR
        / f"checkpoint_step_{step:06d}.pt"
    )


def save_checkpoint(step):
    path = checkpoint_path(
        step
    )

    gate = float(
        model.regional_head
        .attn_gate
        .detach()
        .cpu()
    )

    torch.save({
        "step": int(step),
        "model_state_dict": {
            k: v.detach().cpu()
            for k, v
            in model.state_dict().items()
        },
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "architecture": (
            "bpe200_exp020_signed_positional_cross_attention_32_only_dmlsafe"
        ),
        "region_widths": (
            32,
        ),
        "regional_pooling": (
            "mean_plus_rms"
        ),
        "bpe_merges": BPE_MERGES,
        "merge_hash": MERGE_HASH,
        "cache_format_version": (
            CACHE_FORMAT_VERSION
        ),
        "cache_engine": CACHE_ENGINE,
        "global_flank": GLOBAL_FLANK,
        "global_length": GLOBAL_LENGTH,
        "global_token_rf": GLOBAL_TOKEN_RF,
        "context_stride_bp": (
            ATTN_CONTEXT_STRIDE
        ),
        "context_positions": (
            N_CONTEXT_POSITIONS
        ),
        "attention_dim": ATTN_DIM,
        "attention_heads": ATTN_HEADS,
        "relative_bias_bin_bp": (
            REL_BIAS_BIN_BP
        ),
        "relative_bias_max_bp": (
            REL_BIAS_MAX_BP
        ),
        "relative_bias_buckets": (
            REL_BIAS_BUCKETS
        ),
        "attention_gate": gate,
        "relative_bias_std": float(
            model.regional_head
            .attention
            .relative_bias
            .weight
            .detach()
            .float()
            .std()
            .cpu()
        ),
        "relative_bias_max_abs": float(
            model.regional_head
            .attention
            .relative_bias
            .weight
            .detach()
            .float()
            .abs()
            .max()
            .cpu()
        ),
        "attention_source": (
            "shallow_bpe_stem_on_regular_bp_grid"
        ),
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "training_budget": STEPS,
        "ema_32": float(
            ema32
        ),
        "elapsed_minutes": float(
            (
                time.perf_counter()
                - started
            )
            / 60.0
        ),
    }, path)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print()
    print(
        f"=== CHECKPOINT SAVED @ {step:,} ==="
    )
    print(
        "EMA32:",
        round(
            float(ema32),
            6,
        ),
    )
    print(
        "Attention gate:",
        round(
            gate,
            6,
        ),
    )
    print(
        "Path:",
        path,
    )
    print()


with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    for step in range(
        1,
        STEPS + 1,
    ):
        t0 = time.perf_counter()

        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        gb = build_cached_both_batch(
            batch.tiles
        )

        prep_seconds += (
            time.perf_counter()
            - t0
        )

        plus_exact = (
            batch.plus_target
            .contiguous()
        )

        minus_exact = (
            batch.minus_target_reverse
            .contiguous()
        )

        plus_base_w = (
            batch.plus_loss_weight
            .contiguous()
        )

        minus_base_w = (
            batch.minus_loss_weight_reverse
            .contiguous()
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = forward_both(
            model,
            gb,
            grad=True,
        )

        pp, mp = outputs[32]

        plus_target_r, plus_valid_r = (
            make_region_targets_cpu(
                plus_exact,
                plus_base_w,
                32,
            )
        )

        minus_target_r, minus_valid_r = (
            make_region_targets_cpu(
                minus_exact,
                minus_base_w,
                32,
            )
        )

        plus_loss, plus_pw = (
            stable_region_bce(
                pp,
                plus_target_r.to(device),
                plus_valid_r.to(device),
            )
        )

        minus_loss, minus_pw = (
            stable_region_bce(
                mp,
                minus_target_r.to(device),
                minus_valid_r.to(device),
            )
        )

        loss = (
            plus_loss
            + minus_loss
        ) / 2.0

        value = loss.item()

        if not math.isfinite(
            value
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        loss.backward()
        optimizer.step()

        if ema32 is None:
            ema32 = value
        else:
            ema32 = (
                0.95 * ema32
                + 0.05 * value
            )

        seen += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        gate = float(
            model.regional_head
            .attn_gate
            .detach()
            .cpu()
        )

        relative_bias_weight = (
            model.regional_head
            .attention
            .relative_bias
            .weight
            .detach()
            .float()
        )

        relative_bias_std = float(
            relative_bias_weight
            .std()
            .cpu()
        )

        relative_bias_max_abs = float(
            relative_bias_weight
            .abs()
            .max()
            .cpu()
        )

        row = {
            "step": step,
            "loss_32": value,
            "ema_32": ema32,
            "pos_weight_32": (
                plus_pw
                + minus_pw
            ) / 2.0,
            "attention_gate": gate,
            "relative_bias_std": (
                relative_bias_std
            ),
            "relative_bias_max_abs": (
                relative_bias_max_abs
            ),
            "positions_per_second": (
                seen / elapsed
            ),
            "prep_fraction": (
                prep_seconds / elapsed
            ),
        }

        history.append(
            row
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            print(
                f"step={step:6d} "
                f"loss32={value:.4f} "
                f"ema32={ema32:.4f} "
                f"pw32={row['pos_weight_32']:.1f} "
                f"attn_gate={gate:+.4f} "
                f"relbias_std={relative_bias_std:.4f} "
                f"relbias_max={relative_bias_max_abs:.4f} "
                f"positions/s="
                f"{seen / elapsed:,.0f} "
                f"prep="
                f"{100 * prep_seconds / elapsed:.1f}%"
            )

        if (
            step % SAVE_EVERY == 0
            or step == STEPS
        ):
            save_checkpoint(
                step
            )


fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(w.message).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(
            fallback
        )
    )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print(
    "Elapsed min:",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60,
        2,
    ),
)

print(
    "Final EMA32:",
    ema32,
)

print(
    "Final attention gate:",
    float(
        model.regional_head
        .attn_gate
        .detach()
        .cpu()
    ),
)

print(
    "Final checkpoint:",
    checkpoint_path(
        STEPS
    ),
)


step=     1 loss32=1.0908 ema32=1.0908 pw32=28.1 attn_gate=+0.0990 relbias_std=0.0009 relbias_max=0.0010 positions/s=239,031 prep=14.5%
step=   100 loss32=1.1178 ema32=1.1715 pw32=32.0 attn_gate=+0.0825 relbias_std=0.0129 relbias_max=0.0515 positions/s=247,263 prep=16.7%
step=   200 loss32=1.2412 ema32=1.1126 pw32=23.0 attn_gate=+0.0772 relbias_std=0.0170 relbias_max=0.0793 positions/s=253,889 prep=17.1%
step=   300 loss32=1.2677 ema32=1.0377 pw32=16.3 attn_gate=+0.0706 relbias_std=0.0186 relbias_max=0.0833 positions/s=257,098 prep=17.1%
step=   400 loss32=1.0552 ema32=1.0084 pw32=23.8 attn_gate=+0.0681 relbias_std=0.0200 relbias_max=0.0864 positions/s=258,515 prep=17.2%
step=   500 loss32=0.5812 ema32=1.0246 pw32=32.0 attn_gate=+0.0660 relbias_std=0.0210 relbias_max=0.0889 positions/s=261,417 prep=17.2%
step=   600 loss32=0.9214 ema32=0.9933 pw32=20.4 attn_gate=+0.0660 relbias_std=0.0233 relbias_max=0.1105 positions/s=261,622 prep=17.3%
step=   700 loss32=0.8434 ema32=0.9789 pw32=32.0

RuntimeError: DirectML fallback: The operator 'aten::std.correction' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)

In [16]:
# ============================================================
# FULL HOLDOUT — EXP029 POSITIONAL CROSS-ATTENTION, 32-BP ONLY
# ============================================================
# No sampling and no torch.linspace here.
# This is exactly the complete validation tile index.

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Tiles:", len(validation_tiles))
print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)
print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [17]:
# ============================================================
# FULL HOLDOUT VALIDATION
# REGIONAL AP + TSS COVERAGE / RECALL
# ============================================================

import numpy as np
from scipy.special import expit


TOP_REGION_FRACTIONS = (
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50,
)

TARGET_TSS_RECALLS = (
    0.90,
    0.95,
    0.99,
)


def exact_average_precision(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=bool,
    )

    positives = int(
        labels.sum()
    )

    if positives == 0:
        return np.nan

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y = labels[
        order
    ].astype(
        np.int64
    )

    tp = np.cumsum(
        y,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.arange(
            1,
            y.size + 1,
            dtype=np.float64,
        )
    )

    return float(
        precision[
            y == 1
        ].sum()
        / positives
    )


def region_arrays_from_batch(
    score_plus,
    score_minus_reverse,
    mask,
    counts,
    width,
):
    B = mask.shape[0]
    n_regions = TARGET_LENGTH // width

    plus_score = score_plus[
        :,
        0,
        :,
    ]

    minus_score = score_minus_reverse[
        :,
        0,
        ::-1,
    ]

    score = np.stack(
        (
            plus_score,
            minus_score,
        ),
        axis=1,
    )

    m = mask.reshape(
        B,
        2,
        n_regions,
        width,
    )

    c = counts.reshape(
        B,
        2,
        n_regions,
        width,
    )

    valid = m.all(
        axis=-1
    )

    tss_count = (
        (
            (c > 0)
            & m
        )
        .sum(
            axis=-1
        )
        .astype(
            np.int32
        )
    )

    positive = (
        tss_count > 0
    )

    return (
        score,
        positive,
        tss_count,
        valid,
    )


def summarize_region_ranking(
    width,
    scores,
    positive,
    tss_count,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    positive = np.asarray(
        positive,
        dtype=bool,
    )

    tss_count = np.asarray(
        tss_count,
        dtype=np.int64,
    )

    n = int(scores.size)
    positive_regions = int(
        positive.sum()
    )

    total_tss = int(
        tss_count.sum()
    )

    prevalence = (
        positive_regions / n
    )

    ap = exact_average_precision(
        scores,
        positive,
    )

    ap_lift = (
        ap / prevalence
        if prevalence > 0
        else np.nan
    )

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y_region = (
        positive[
            order
        ].astype(
            np.int64
        )
    )

    y_tss = (
        tss_count[
            order
        ].astype(
            np.int64
        )
    )

    cum_positive_regions = (
        np.cumsum(
            y_region,
            dtype=np.int64,
        )
    )

    cum_tss = (
        np.cumsum(
            y_tss,
            dtype=np.int64,
        )
    )

    ranking_rows = []

    for fraction in TOP_REGION_FRACTIONS:
        k = max(
            1,
            int(
                np.ceil(
                    fraction * n
                )
            ),
        )

        k = min(
            k,
            n,
        )

        selected_positive_regions = int(
            cum_positive_regions[
                k - 1
            ]
        )

        covered_tss = int(
            cum_tss[
                k - 1
            ]
        )

        ranking_rows.append({
            "region_width_bp": width,
            "selection_type": "top_fraction",
            "target": fraction,
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": covered_tss,
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    for recall_target in TARGET_TSS_RECALLS:
        required = (
            recall_target
            * total_tss
        )

        idx = int(
            np.searchsorted(
                cum_tss,
                required,
                side="left",
            )
        )

        idx = min(
            idx,
            n - 1,
        )

        k = idx + 1

        selected_positive_regions = int(
            cum_positive_regions[
                idx
            ]
        )

        covered_tss = int(
            cum_tss[
                idx
            ]
        )

        ranking_rows.append({
            "region_width_bp": width,
            "selection_type": "target_tss_recall",
            "target": recall_target,
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": covered_tss,
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    summary = {
        "region_width_bp": width,
        "regions": n,
        "positive_regions": positive_regions,
        "regional_prevalence": prevalence,
        "total_tss": total_tss,
        "regional_average_precision": ap,
        "regional_ap_lift": ap_lift,
    }

    return (
        summary,
        ranking_rows,
    )


def evaluate_regional_model(
    model,
    tiles,
    batch_size=8,
    progress_every=500,
):
    score_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positive_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    tss_count_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positions = 0
    exact_positives = 0

    started = time.perf_counter()

    was_training = model.training
    model.eval()

    try:
        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            for batch_no, start in enumerate(
                range(
                    0,
                    len(tiles),
                    batch_size,
                ),
                1,
            ):
                selected = tiles.iloc[
                    start:
                    start + batch_size
                ]

                batch = (
                    validation_generator
                    .make_batch(
                        selected
                    )
                )

                gb = (
                    build_cached_both_batch(
                        batch.tiles
                    )
                )

                outputs = forward_both(
                    model,
                    gb,
                    grad=False,
                )

                mask = (
                    batch.mask
                    .numpy()
                    .astype(bool)
                )

                counts = (
                    batch.count
                    .numpy()
                )

                positions += int(
                    mask.sum()
                )

                exact_positives += int(
                    (
                        mask
                        & (counts > 0)
                    ).sum()
                )

                for width in REGION_WIDTHS:
                    pp, mp = outputs[
                        width
                    ]

                    plus_score = expit(
                        pp.detach()
                        .cpu()
                        .numpy()
                    )

                    minus_score_reverse = expit(
                        mp.detach()
                        .cpu()
                        .numpy()
                    )

                    (
                        score,
                        positive,
                        tss_count,
                        valid,
                    ) = region_arrays_from_batch(
                        plus_score,
                        minus_score_reverse,
                        mask,
                        counts,
                        width,
                    )

                    score_chunks[
                        width
                    ].append(
                        score[
                            valid
                        ].astype(
                            np.float32
                        )
                    )

                    positive_chunks[
                        width
                    ].append(
                        positive[
                            valid
                        ].astype(
                            np.bool_
                        )
                    )

                    tss_count_chunks[
                        width
                    ].append(
                        tss_count[
                            valid
                        ].astype(
                            np.int16
                        )
                    )

                finished = (
                    start
                    + len(selected)
                    >= len(tiles)
                )

                if (
                    batch_no == 1
                    or batch_no % progress_every == 0
                    or finished
                ):
                    elapsed = (
                        time.perf_counter()
                        - started
                    )

                    print(
                        f"tiles="
                        f"{start + len(selected):>5}"
                        f"/{len(tiles)} "
                        f"positions="
                        f"{positions:>9} "
                        f"positions/s="
                        f"{positions / elapsed:,.0f}"
                    )

        fallback = tuple(
            str(w.message)
            for w in caught
            if "fall back"
            in str(w.message).lower()
        )

    finally:
        model.train(
            was_training
        )

    if fallback:
        raise RuntimeError(
            "DirectML fallback: "
            + " | ".join(fallback)
        )

    assert positions == int(
        tiles[
            "allowed_position_strand"
        ].sum()
    )

    assert exact_positives == int(
        tiles[
            "positive_positions"
        ].sum()
    )

    summary_rows = []
    ranking_rows = []

    for width in REGION_WIDTHS:
        scores = np.concatenate(
            score_chunks[
                width
            ]
        )

        positive = np.concatenate(
            positive_chunks[
                width
            ]
        )

        tss_count = np.concatenate(
            tss_count_chunks[
                width
            ]
        ).astype(
            np.int64
        )

        summary, rows = (
            summarize_region_ranking(
                width,
                scores,
                positive,
                tss_count,
            )
        )

        summary_rows.append(
            summary
        )

        ranking_rows.extend(
            rows
        )

    summary_df = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            "region_width_bp",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    ranking_df = (
        pd.DataFrame(
            ranking_rows
        )
        .sort_values(
            [
                "region_width_bp",
                "selection_type",
                "target",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    return (
        summary_df,
        ranking_df,
        elapsed,
        positions / elapsed,
    )


# ============================================================
# EXP029 FULL HOLDOUT @10K
# ============================================================

EXPECTED_ARCHITECTURE = (
    "bpe200_exp020_signed_positional_cross_attention_32_only_dmlsafe"
)

summary_frames = []
ranking_frames = []
checkpoint_rows = []

for checkpoint_step in MILESTONE_STEPS:
    path = checkpoint_path(
        checkpoint_step
    )

    if not path.exists():
        print(
            "Skip missing checkpoint:",
            path,
        )
        continue

    print()
    print(
        "=" * 72
    )
    print(
        f"FULL HOLDOUT @ {checkpoint_step:,} STEPS"
    )
    print(
        "=" * 72
    )

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    assert (
        checkpoint[
            "architecture"
        ]
        == EXPECTED_ARCHITECTURE
    )

    assert tuple(
        checkpoint[
            "region_widths"
        ]
    ) == (
        32,
    )

    assert (
        checkpoint[
            "bpe_merges"
        ]
        == BPE_MERGES
    )

    assert (
        checkpoint[
            "merge_hash"
        ]
        == MERGE_HASH
    )

    validation_model = (
        create_model()
        .to(
            device
        )
    )

    validation_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    (
        summary_df,
        ranking_df,
        seconds,
        speed,
    ) = evaluate_regional_model(
        validation_model,
        validation_tiles,
        batch_size=8,
        progress_every=500,
    )

    summary_df = summary_df.copy()
    ranking_df = ranking_df.copy()

    summary_df.insert(
        0,
        "step",
        checkpoint_step,
    )

    ranking_df.insert(
        0,
        "step",
        checkpoint_step,
    )

    summary_frames.append(
        summary_df
    )

    ranking_frames.append(
        ranking_df
    )

    checkpoint_rows.append({
        "step": checkpoint_step,
        "attention_gate": float(
            checkpoint[
                "attention_gate"
            ]
        ),
        "relative_bias_std": float(
            checkpoint[
                "relative_bias_std"
            ]
        ),
        "relative_bias_max_abs": float(
            checkpoint[
                "relative_bias_max_abs"
            ]
        ),
        "validation_minutes": (
            seconds / 60.0
        ),
        "validation_positions_per_second": speed,
    })

    print()
    display(
        summary_df
    )

    del validation_model
    del checkpoint
    gc.collect()


if not summary_frames:
    raise RuntimeError(
        "No milestone checkpoints found."
    )

holdout_summary = pd.concat(
    summary_frames,
    ignore_index=True,
)

holdout_ranking = pd.concat(
    ranking_frames,
    ignore_index=True,
)

checkpoint_info = pd.DataFrame(
    checkpoint_rows
)

top10 = (
    holdout_ranking[
        (
            holdout_ranking[
                "selection_type"
            ]
            == "top_fraction"
        )
        & np.isclose(
            holdout_ranking[
                "target"
            ],
            0.10,
        )
    ][
        [
            "step",
            "tss_recall",
            "regional_precision",
        ]
    ]
    .copy()
)

need95 = (
    holdout_ranking[
        (
            holdout_ranking[
                "selection_type"
            ]
            == "target_tss_recall"
        )
        & np.isclose(
            holdout_ranking[
                "target"
            ],
            0.95,
        )
    ][
        [
            "step",
            "selected_fraction",
        ]
    ]
    .copy()
)

learning_curve = (
    holdout_summary[
        [
            "step",
            "regional_prevalence",
            "regional_average_precision",
            "regional_ap_lift",
        ]
    ]
    .merge(
        top10,
        on="step",
    )
    .merge(
        need95,
        on="step",
    )
    .merge(
        checkpoint_info,
        on="step",
    )
    .rename(
        columns={
            "regional_average_precision": "AP32",
            "regional_ap_lift": "AP32_lift",
            "tss_recall": "TSS_recall_top10",
            "regional_precision": "precision_top10",
            "selected_fraction": "regions_for_95pct_TSS",
        }
    )
    .sort_values(
        "step"
    )
    .reset_index(
        drop=True
    )
)

SUMMARY_PATH = (
    RUN_DIR
    / "holdout_summary_milestones.csv"
)

RANKING_PATH = (
    RUN_DIR
    / "holdout_ranking_milestones.csv"
)

LEARNING_CURVE_PATH = (
    RUN_DIR
    / "learning_curve_32bp.csv"
)

holdout_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

holdout_ranking.to_csv(
    RANKING_PATH,
    index=False,
)

learning_curve.to_csv(
    LEARNING_CURVE_PATH,
    index=False,
)

print()
print(
    "=== EXP029 32-BP LEARNING CURVE ==="
)

display(
    learning_curve
)


# ============================================================
# ATTENTION CONTEXT DIAGNOSTIC
# Best checkpoint by full-holdout AP32
# ============================================================

best_row = (
    learning_curve
    .sort_values(
        "AP32",
        ascending=False,
    )
    .iloc[0]
)

BEST_STEP = int(
    best_row[
        "step"
    ]
)

print()
print(
    "Best checkpoint by AP32:",
    BEST_STEP,
)

best_checkpoint = torch.load(
    checkpoint_path(
        BEST_STEP
    ),
    map_location="cpu",
    weights_only=False,
)

attention_model = (
    create_model()
    .to(
        device
    )
)

attention_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

attention_model.eval()

# Deterministic broad sample across the full holdout.
ATTENTION_DIAGNOSTIC_TILES = min(
    2000,
    len(validation_tiles),
)

diag_indices = np.linspace(
    0,
    len(validation_tiles) - 1,
    ATTENTION_DIAGNOSTIC_TILES,
    dtype=np.int64,
)

diagnostic_tiles = (
    validation_tiles
    .iloc[
        diag_indices
    ]
    .reset_index(
        drop=True
    )
)

CONTEXT_RADII_BP = np.asarray(
    [
        128,
        256,
        512,
        1024,
        2048,
        4096,
        6144,
    ],
    dtype=np.float32,
)

EFFECTIVE_MASS_TARGETS = (
    0.50,
    0.80,
    0.90,
    0.95,
)

context_positions = (
    CONTEXT_POSITIONS_BP
    .cpu()
    .numpy()
    .astype(
        np.float32
    )
)

region_centers = (
    GLOBAL_FLANK
    + (
        np.arange(
            N_REGIONS,
            dtype=np.float32,
        )
        + 0.5
    )
    * REGION_WIDTH
)

distance_matrix = np.abs(
    region_centers[
        :,
        None,
    ]
    - context_positions[
        None,
        :,
    ]
)

signed_distance_matrix = (
    context_positions[
        None,
        :,
    ]
    - region_centers[
        :,
        None,
    ]
)

radius_mass_records = []
effective_radius_records = []
uniform_effective_radius_records = []
entropy_records = []
edge_mass_records = []
left_right_records = []

positive_queries = 0
all_queries = 0

for batch_no, start in enumerate(
    range(
        0,
        len(diagnostic_tiles),
        4,
    ),
    1,
):
    selected = diagnostic_tiles.iloc[
        start:
        start + 4
    ]

    batch = (
        validation_generator
        .make_batch(
            selected
        )
    )

    gb = build_cached_both_batch(
        batch.tiles
    )

    (
        _,
        attention,
    ) = forward_both_with_attention(
        attention_model,
        gb,
    )

    # Average heads only for interpretability.
    # Shape: [2B, 64, S]
    attn = (
        attention
        .mean(
            dim=1
        )
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float64
        )
    )

    B = len(selected)

    counts = (
        batch.count
        .numpy()
    )

    mask = (
        batch.mask
        .numpy()
        .astype(bool)
    )

    plus_counts = counts[
        :,
        0,
        :,
    ]

    minus_counts_reverse = counts[
        :,
        1,
        ::-1,
    ]

    plus_mask = mask[
        :,
        0,
        :,
    ]

    minus_mask_reverse = mask[
        :,
        1,
        ::-1,
    ]

    oriented_counts = np.concatenate(
        (
            plus_counts,
            minus_counts_reverse,
        ),
        axis=0,
    )

    oriented_mask = np.concatenate(
        (
            plus_mask,
            minus_mask_reverse,
        ),
        axis=0,
    )

    positive_region = (
        (
            oriented_counts > 0
        )
        & oriented_mask
    ).reshape(
        2 * B,
        N_REGIONS,
        REGION_WIDTH,
    ).any(
        axis=-1
    )

    valid_region = (
        oriented_mask.reshape(
            2 * B,
            N_REGIONS,
            REGION_WIDTH,
        )
        .all(
            axis=-1
        )
    )

    # Real context validity for entropy denominator / edge interpretation.
    context_valid = (
        gb[
            "context_valid"
        ]
        .numpy()
        .astype(bool)
    )

    for row in range(
        2 * B
    ):
        valid_context_count = max(
            2,
            int(
                context_valid[
                    row
                ].sum()
            ),
        )

        log_norm = np.log(
            valid_context_count
        )

        for r in range(
            N_REGIONS
        ):
            if not valid_region[
                row,
                r,
            ]:
                continue

            all_queries += 1

            if not positive_region[
                row,
                r,
            ]:
                continue

            positive_queries += 1

            p = attn[
                row,
                r,
                :
            ]

            # Re-normalize tiny numerical drift after CPU conversion.
            p = p / max(
                p.sum(),
                1e-12,
            )

            distances = distance_matrix[
                r
            ]

            signed = signed_distance_matrix[
                r
            ]

            valid_context = (
                context_valid[
                    row
                ].astype(
                    np.float64
                )
            )

            uniform_p = (
                valid_context
                / max(
                    valid_context.sum(),
                    1.0,
                )
            )

            for radius in CONTEXT_RADII_BP:
                in_radius = (
                    distances
                    <= radius
                )

                radius_mass_records.append({
                    "radius_bp": int(
                        radius
                    ),
                    "mass": float(
                        p[
                            in_radius
                        ].sum()
                    ),
                    "uniform_mass": float(
                        uniform_p[
                            in_radius
                        ].sum()
                    ),
                })

            order = np.argsort(
                distances,
                kind="mergesort",
            )

            sorted_dist = distances[
                order
            ]

            sorted_mass = p[
                order
            ]

            sorted_uniform_mass = (
                uniform_p[
                    order
                ]
            )

            cum = np.cumsum(
                sorted_mass
            )

            uniform_cum = np.cumsum(
                sorted_uniform_mass
            )

            for target_mass in EFFECTIVE_MASS_TARGETS:
                idx = int(
                    np.searchsorted(
                        cum,
                        target_mass,
                        side="left",
                    )
                )

                idx = min(
                    idx,
                    len(
                        sorted_dist
                    ) - 1,
                )

                effective_radius_records.append({
                    "target_mass": float(
                        target_mass
                    ),
                    "radius_bp": float(
                        sorted_dist[
                            idx
                        ]
                    ),
                })

                uniform_idx = int(
                    np.searchsorted(
                        uniform_cum,
                        target_mass,
                        side="left",
                    )
                )

                uniform_idx = min(
                    uniform_idx,
                    len(
                        sorted_dist
                    ) - 1,
                )

                uniform_effective_radius_records.append({
                    "target_mass": float(
                        target_mass
                    ),
                    "radius_bp": float(
                        sorted_dist[
                            uniform_idx
                        ]
                    ),
                })

            positive_p = p[
                p > 0
            ]

            entropy = -float(
                (
                    positive_p
                    * np.log(
                        positive_p
                    )
                ).sum()
            )

            entropy_records.append(
                entropy
                / log_norm
            )

            edge_mask = (
                (
                    context_positions
                    < 512
                )
                | (
                    context_positions
                    >= (
                        GLOBAL_LENGTH
                        - 512
                    )
                )
            )

            edge_mass_records.append(
                float(
                    p[
                        edge_mask
                    ].sum()
                )
            )

            left_right_records.append({
                "left_mass": float(
                    p[
                        signed < 0
                    ].sum()
                ),
                "right_mass": float(
                    p[
                        signed > 0
                    ].sum()
                ),
            })

    if (
        batch_no == 1
        or batch_no % 100 == 0
        or start + len(selected)
        >= len(diagnostic_tiles)
    ):
        print(
            f"attention diagnostic tiles="
            f"{start + len(selected):>5}"
            f"/{len(diagnostic_tiles)} "
            f"positive queries="
            f"{positive_queries:,}"
        )


radius_mass_df = pd.DataFrame(
    radius_mass_records
)

effective_radius_df = pd.DataFrame(
    effective_radius_records
)

left_right_df = pd.DataFrame(
    left_right_records
)

radius_summary = (
    radius_mass_df
    .groupby(
        "radius_bp",
        as_index=False,
    )
    .agg(
        mean_mass=(
            "mass",
            "mean",
        ),
        median_mass=(
            "mass",
            "median",
        ),
        uniform_mean_mass=(
            "uniform_mass",
            "mean",
        ),
    )
)

radius_summary[
    "enrichment_vs_uniform"
] = (
    radius_summary[
        "mean_mass"
    ]
    / radius_summary[
        "uniform_mean_mass"
    ]
)

effective_summary = (
    effective_radius_df
    .groupby(
        "target_mass",
        as_index=False,
    )
    .agg(
        median_radius_bp=(
            "radius_bp",
            "median",
        ),
        mean_radius_bp=(
            "radius_bp",
            "mean",
        ),
        p90_radius_bp=(
            "radius_bp",
            lambda x: np.quantile(
                x,
                0.90,
            ),
        ),
    )
)


uniform_effective_radius_df = (
    pd.DataFrame(
        uniform_effective_radius_records
    )
)

uniform_effective_summary = (
    uniform_effective_radius_df
    .groupby(
        "target_mass",
        as_index=False,
    )
    .agg(
        uniform_median_radius_bp=(
            "radius_bp",
            "median",
        ),
    )
)

effective_summary = (
    effective_summary
    .merge(
        uniform_effective_summary,
        on="target_mass",
        how="left",
    )
)

effective_summary[
    "radius_ratio_vs_uniform"
] = (
    effective_summary[
        "median_radius_bp"
    ]
    / effective_summary[
        "uniform_median_radius_bp"
    ]
)

context_summary = pd.DataFrame([{
    "best_step": BEST_STEP,
    "diagnostic_tiles": len(
        diagnostic_tiles
    ),
    "positive_queries": (
        positive_queries
    ),
    "attention_gate": float(
        best_checkpoint[
            "attention_gate"
        ]
    ),
    "normalized_attention_entropy_mean": float(
        np.mean(
            entropy_records
        )
    ),
    "normalized_attention_entropy_median": float(
        np.median(
            entropy_records
        )
    ),
    "edge_512bp_mass_mean": float(
        np.mean(
            edge_mass_records
        )
    ),
    "edge_512bp_mass_median": float(
        np.median(
            edge_mass_records
        )
    ),
    "left_mass_mean": float(
        left_right_df[
            "left_mass"
        ].mean()
    ),
    "right_mass_mean": float(
        left_right_df[
            "right_mass"
        ].mean()
    ),
}])

# ============================================================
# LEARNED SIGNED RELATIVE-POSITION BIAS
# ============================================================

relative_bias_weight = (
    attention_model
    .regional_head
    .attention
    .relative_bias
    .weight
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.float64
    )
)

bucket_ids = np.arange(
    REL_BIAS_BUCKETS,
    dtype=np.int64,
)

signed_bin_center_bp = np.zeros(
    REL_BIAS_BUCKETS,
    dtype=np.float64,
)

for bucket_id in bucket_ids:
    if bucket_id < REL_BIAS_CENTER:
        magnitude = (
            REL_BIAS_CENTER
            - bucket_id
        )

        signed_bin_center_bp[
            bucket_id
        ] = -(
            magnitude
            - 0.5
        ) * REL_BIAS_BIN_BP

    elif bucket_id > REL_BIAS_CENTER:
        magnitude = (
            bucket_id
            - REL_BIAS_CENTER
        )

        signed_bin_center_bp[
            bucket_id
        ] = (
            magnitude
            - 0.5
        ) * REL_BIAS_BIN_BP

relative_bias_table = pd.DataFrame({
    "bucket_id": bucket_ids,
    "signed_distance_center_bp": (
        signed_bin_center_bp
    ),
})

for head in range(
    ATTN_HEADS
):
    relative_bias_table[
        f"head_{head}"
    ] = relative_bias_weight[
        :,
        head,
    ]

print()
print(
    "=== LEARNED SIGNED RELATIVE-POSITION BIAS ==="
)

display(
    relative_bias_table
)

print()
print(
    "=== ATTENTION MASS WITHIN QUERY-CENTERED RADIUS ==="
)

display(
    radius_summary
)

print()
print(
    "=== EFFECTIVE CONTEXT RADIUS ==="
)

display(
    effective_summary
)

print()
print(
    "=== CONTEXT SUMMARY ==="
)

display(
    context_summary
)

RADIUS_PATH = (
    RUN_DIR
    / "attention_mass_by_radius_best_checkpoint.csv"
)

EFFECTIVE_PATH = (
    RUN_DIR
    / "attention_effective_radius_best_checkpoint.csv"
)

CONTEXT_SUMMARY_PATH = (
    RUN_DIR
    / "attention_context_summary_best_checkpoint.csv"
)


RELATIVE_BIAS_PATH = (
    RUN_DIR
    / "learned_signed_relative_position_bias_best_checkpoint.csv"
)

radius_summary.to_csv(
    RADIUS_PATH,
    index=False,
)

effective_summary.to_csv(
    EFFECTIVE_PATH,
    index=False,
)

context_summary.to_csv(
    CONTEXT_SUMMARY_PATH,
    index=False,
)

relative_bias_table.to_csv(
    RELATIVE_BIAS_PATH,
    index=False,
)

print()
print(
    "Saved:",
    LEARNING_CURVE_PATH,
)

print(
    "Saved:",
    RADIUS_PATH,
)

print(
    "Saved:",
    EFFECTIVE_PATH,
)

print(
    "Saved:",
    CONTEXT_SUMMARY_PATH,
)

print(
    "Saved:",
    RELATIVE_BIAS_PATH,
)



FULL HOLDOUT @ 10,000 STEPS
tiles=    8/24700 positions=    23524 positions/s=617,853
tiles= 4000/24700 positions= 13735464 positions/s=1,262,024
tiles= 8000/24700 positions= 27763582 positions/s=1,312,666
tiles=12000/24700 positions= 41193676 positions/s=1,313,895
tiles=16000/24700 positions= 54896176 positions/s=1,323,347
tiles=20000/24700 positions= 67811662 positions/s=1,307,288
tiles=24000/24700 positions= 81734314 positions/s=1,295,787
tiles=24700/24700 positions= 83807486 positions/s=1,290,913



,step,region_width_bp,regions,positive_regions,regional_prevalence,total_tss,regional_average_precision,regional_ap_lift
0,10000,32,2560380,27982,0.010929,55730,0.102162,9.34788



=== EXP029 32-BP LEARNING CURVE ===


,step,regional_prevalence,AP32,AP32_lift,TSS_recall_top10,precision_top10,regions_for_95pct_TSS,attention_gate,relative_bias_std,relative_bias_max_abs,validation_minutes,validation_positions_per_second
0,10000,0.010929,0.102162,9.34788,0.641683,0.05621,0.5097,0.114893,0.329734,1.037568,1.092921,1.278036e+06



Best checkpoint by AP32: 10000
attention diagnostic tiles=    4/2000 positive queries=0
attention diagnostic tiles=  400/2000 positive queries=523
attention diagnostic tiles=  800/2000 positive queries=1,023
attention diagnostic tiles= 1200/2000 positive queries=1,362
attention diagnostic tiles= 1600/2000 positive queries=1,863
attention diagnostic tiles= 2000/2000 positive queries=2,294

=== LEARNED SIGNED RELATIVE-POSITION BIAS ===


...


=== EFFECTIVE CONTEXT RADIUS ===


,target_mass,median_radius_bp,mean_radius_bp,p90_radius_bp,uniform_median_radius_bp,radius_ratio_vs_uniform
0,0.50,2712.0,2724.491718,3048.0,3576.0,0.758389
1,0.80,4936.0,4943.734961,5304.0,5736.0,0.860530
2,0.90,5880.0,5876.533566,6120.0,6456.0,0.910781
3,0.95,6408.0,6450.015693,6776.0,6984.0,0.917526



=== CONTEXT SUMMARY ===


,best_step,diagnostic_tiles,positive_queries,attention_gate,normalized_attention_entropy_mean,normalized_attention_entropy_median,edge_512bp_mass_mean,edge_512bp_mass_median,left_mass_mean,right_mass_mean
0,10000,2000,2294,0.114893,0.883477,0.888495,0.037883,0.035729,0.472109,0.527891



Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY\run_001\learning_curve_32bp.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY\run_001\attention_mass_by_radius_best_checkpoint.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY\run_001\attention_effective_radius_best_checkpoint.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY\run_001\attention_context_summary_best_checkpoint.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP029_BPE200_POSITIONAL_CROSS_ATTENTION_32_ONLY\run_001\learned_signed_relative_position_bias_best_checkpoint.csv
